# 05 — Shor Code, CSS Codes, and the Steane Code

## Read before you begin

- Arthur Pesah, [*Stabilizer Formalism I*](https://arthurpesah.me/blog/2023-01-31-stabilizer-formalism-1/).
- Sections to read: **Our first truly quantum code: Shor's code**, **Stabilizer formalism: first definitions**, **Quantum codes from classical codes: the CSS construction**, **Steane code, the quantum version of the Hamming code**.
- Concepts to extract: how Shor's code nests a bit-flip code inside a phase-flip code; the general definition of a stabilizer group and code space (beyond the two-generator repetition-code case from notebook 03); the CSS recipe for turning classical parity checks into commuting quantum stabilizers; why the Hamming code specifically can generate *both* the $X$-type and $Z$-type stabilizers of a quantum code.

This notebook does not re-derive what a stabilizer or a syndrome is — notebook 03 already covers that. It builds directly on those definitions at a larger scale.

## Learning objectives

By the end of this notebook you should be able to:

1. construct Shor's $[[9,1,3]]$ code's block states $|0_L\rangle$, $|1_L\rangle$ and its 8 stabilizer generators;
2. detect and correct a representative bit-flip error ($X_4$) and phase-flip error ($Z_4$);
3. explain *degeneracy* — why several different physical errors demand the identical correction — using the Shor code as a concrete example;
4. verify computationally that the Shor code is $[[9,1,3]]$: 1 logical qubit from 9 physical qubits, distance 3;
5. state the general CSS construction and explain why it needs a classical code whose parity checks are pairwise orthogonal;
6. build the Steane code's $X$- and $Z$-type stabilizers directly from the $[7,4,3]$ Hamming code's parity-check matrix (notebook 04), verify they commute, and read off single-error syndromes.

**Expected outputs:** verified stabilizer/state constructions for both codes, an explicit degeneracy demonstration, a computational proof of $[[9,1,3]]$ for Shor's code, and a Steane-code syndrome table built the same way notebook 04 built the classical one.

**Assumes:** notebook 03 (stabilizers, syndromes, eigenvalues) and notebook 04 (the $[7,4,3]$ Hamming code's $H$ matrix, reused here verbatim).

## 1. Pauli machinery for many qubits

We need $X$, $Y$, $Z$ this time (Shor code's distance proof in Section 4 needs to rule out *every* single- and two-qubit Pauli error, not just $X$ and $Z$ separately), embedded into a 9-qubit (and later 7-qubit) space.

In [1]:
import numpy as np

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {"I": I2, "X": X, "Y": Y, "Z": Z}

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)


def kron_all(vectors_or_matrices):
    result = vectors_or_matrices[0]
    for m in vectors_or_matrices[1:]:
        result = np.kron(result, m)
    return result


def pauli_string_matrix(string):
    """Build the full-space matrix for a Pauli string like 'ZZIIIIIII'."""
    return kron_all([PAULI[ch] for ch in string])


def basis_state(bits):
    """The computational basis state |bits> as a state vector."""
    return kron_all([ket1 if b else ket0 for b in bits])


print(pauli_string_matrix("XZ").shape)  # sanity check: 2-qubit Pauli string -> 4x4

(4, 4)


## 2. Shor's code: nesting bit-flip protection inside phase-flip protection

Shor's code arranges 9 physical qubits into 3 blocks of 3. Each block is a 3-qubit **GHZ-like pair** of basis states, $|000\rangle \pm |111\rangle$, and the logical states are a *product* of three such blocks, all "+" for $0_L$ and all "−" for $1_L$:

$$|0_L\rangle = \frac{1}{2\sqrt{2}}\big(|000\rangle + |111\rangle\big)^{\otimes 3} \qquad |1_L\rangle = \frac{1}{2\sqrt{2}}\big(|000\rangle - |111\rangle\big)^{\otimes 3}$$

The inner structure (3 qubits per block) is exactly notebook 02's repetition code, protecting against bit flips ($X$ errors) within a block. The outer structure (comparing the $\pm$ sign across the 3 blocks) is a repetition code *for the sign*, protecting against phase flips ($Z$ errors) — a $Z$ error inside a block flips that block's sign from $+$ to $-$, exactly the way an $X$ error flips a classical bit.

In [2]:
block_plus = basis_state((0, 0, 0)) + basis_state((1, 1, 1))
block_minus = basis_state((0, 0, 0)) - basis_state((1, 1, 1))

state_0L = kron_all([block_plus, block_plus, block_plus])
state_0L = state_0L / np.linalg.norm(state_0L)

state_1L = kron_all([block_minus, block_minus, block_minus])
state_1L = state_1L / np.linalg.norm(state_1L)

print("dimension of the 9-qubit space:", state_0L.shape[0])
print("|0_L> and |1_L> are orthogonal:", np.isclose(np.vdot(state_0L, state_1L), 0))
print("|0_L> is normalized:", np.isclose(np.linalg.norm(state_0L), 1))

dimension of the 9-qubit space: 512
|0_L> and |1_L> are orthogonal: True
|0_L> is normalized: True


## 3. The 8 stabilizer generators

$n - k = 9 - 1 = 8$ independent generators are needed. Six are **$Z$-type, within a block** (the same $Z_iZ_j$ parity checks as notebook 03's repetition code, one pair per block); two are **$X$-type, spanning two blocks at a time**:

In [3]:
shor_generator_strings = [
    "ZZIIIIIII",  # block A: Z0 Z1
    "IZZIIIIII",  # block A: Z1 Z2
    "IIIZZIIII",  # block B: Z3 Z4
    "IIIIZZIII",  # block B: Z4 Z5
    "IIIIIIZZI",  # block C: Z6 Z7
    "IIIIIIIZZ",  # block C: Z7 Z8
    "XXXXXXIII",  # blocks A+B: X0..X5
    "IIIXXXXXX",  # blocks B+C: X3..X8
]
shor_generators = [pauli_string_matrix(s) for s in shor_generator_strings]

for s, S in zip(shor_generator_strings, shor_generators):
    e0 = np.vdot(state_0L, S @ state_0L).real
    e1 = np.vdot(state_1L, S @ state_1L).real
    print(f"{s}  eigenvalue on |0_L>: {e0:+.1f}   eigenvalue on |1_L>: {e1:+.1f}")

ZZIIIIIII  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IZZIIIIII  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IIIZZIIII  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IIIIZZIII  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IIIIIIZZI  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IIIIIIIZZ  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
XXXXXXIII  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0
IIIXXXXXX  eigenvalue on |0_L>: +1.0   eigenvalue on |1_L>: +1.0


Both logical states are $+1$ eigenstates of all 8 generators, exactly as Section 1's definition of a stabilizer requires — this is the 9-qubit version of notebook 03's `Z0Z1`/`Z1Z2` check on `|000>`/`|111>`.

### Prediction 1

Qubit 4 (0-indexed) is the middle qubit of block B. **Before running the next cell:** an $X_4$ error should flip exactly the $Z$-type checks that touch qubit 4 — which two of the six? An $X$ error never flips an $X$-type check (same-letter Paulis always commute) — so what should the two $X$-type stabilizer eigenvalues be after $X_4$? Now do the same reasoning for a $Z_4$ error, but for the $X$-type checks instead.

In [4]:
def eigenvalues_after_error(error_string, logical_state):
    E = pauli_string_matrix(error_string)
    corrupted = E @ logical_state
    return [np.vdot(corrupted, S @ corrupted).real for S in shor_generators]


x4_error = "IIIIXIIII"
z4_error = "IIIIZIIII"

print("Generators:      ", shor_generator_strings)
print("Eigenvalues after X4:", [f"{v:+.0f}" for v in eigenvalues_after_error(x4_error, state_0L)])
print("Eigenvalues after Z4:", [f"{v:+.0f}" for v in eigenvalues_after_error(z4_error, state_0L)])

Generators:       ['ZZIIIIIII', 'IZZIIIIII', 'IIIZZIIII', 'IIIIZZIII', 'IIIIIIZZI', 'IIIIIIIZZ', 'XXXXXXIII', 'IIIXXXXXX']
Eigenvalues after X4: ['+1', '+1', '-1', '-1', '+1', '+1', '+1', '+1']
Eigenvalues after Z4: ['+1', '+1', '+1', '+1', '+1', '+1', '-1', '-1']


As predicted: $X_4$ flips only the two $Z$-type checks that include qubit 4 (`IIIZZIIII` and `IIIIZZIII`, block B's internal checks) — the $X$-type checks are untouched because $X$ always commutes with $X$. $Z_4$ does the opposite: it flips both $X$-type checks (both of them include qubit 4, since block B sits inside both 6-qubit spans) and leaves every $Z$-type check alone. This reproduces notebook 02/03's 2-bit-syndrome pattern, but now applied to *blocks* rather than individual bits: the $X$-type syndrome $(s_{AB}, s_{BC})$ tells you which block has a phase error exactly the way $(s_0, s_1)$ told you which bit had flipped in the repetition code.

## 4. Degeneracy: why a block-level correction is enough

A $Z$ error can land on *any* of the 3 qubits in a block. Does it matter which one, for the purpose of correcting it?

In [5]:
corrupted_by_z0 = pauli_string_matrix("ZIIIIIIII") @ state_0L
corrupted_by_z1 = pauli_string_matrix("IZIIIIIII") @ state_0L
corrupted_by_z2 = pauli_string_matrix("IIZIIIIII") @ state_0L

same_as_01 = np.allclose(corrupted_by_z0, corrupted_by_z1)
same_as_02 = np.allclose(corrupted_by_z0, corrupted_by_z2)
print("Z0 and Z1 produce the identical corrupted state:", same_as_01)
print("Z0 and Z2 produce the identical corrupted state:", same_as_02)

Z0 and Z1 produce the identical corrupted state: True
Z0 and Z2 produce the identical corrupted state: True


$Z_0$, $Z_1$, and $Z_2$ don't just give the same *syndrome* — they produce the exact same resulting quantum state. This is **degeneracy**: multiple distinct physical errors are, as far as the code is concerned, the same error. A decoder never needs to figure out *which* qubit in block A had the phase flip — it only needs to know *that* block A did, and can apply a fixed, arbitrary representative correction (say, always $Z_0$) to fully repair the state. This is different from the Hamming code in notebook 04, where the syndrome pinpointed one specific bit out of seven; here, an entire block of physical qubits maps to one syndrome outcome, and that's not a deficiency — it's *why* fewer syndrome bits (2, for 3 blocks) are needed than physical qubits-at-risk (9).

## 5. Proving Shor's code is $[[9,1,3]]$

The notation $[[n,k,d]]$ (double brackets, for quantum codes) means: $n=9$ physical qubits, $k$ logical qubits, distance $d$. We check both $k=1$ and $d=3$ computationally rather than asserting them.

**$k=1$:** the joint $+1$-eigenspace of all 8 generators is spanned by the projector $P = \prod_i \frac{I + S_i}{2}$. Its rank is $2^{n-\text{(number of generators)}} = 2^{9-8} = 2^k$.

In [6]:
dim = 512
identity = np.eye(dim, dtype=complex)
projector = identity.copy()
for S in shor_generators:
    projector = projector @ ((identity + S) / 2)

rank = np.linalg.matrix_rank(projector, tol=1e-8)
print("Rank of the joint +1-eigenspace projector:", rank, " (expected 2^k = 2, so k = 1)")

Rank of the joint +1-eigenspace projector: 2  (expected 2^k = 2, so k = 1)


**$d = 3$:** we already have one weight-3 logical operator on hand. Recall $Z_0Z_3Z_6$ (one $Z$ per block) should flip every block's sign, mapping $|0_L\rangle \leftrightarrow |1_L\rangle$ — that's a logical $\bar{X}$. And $X_0X_1X_2$ (all of block A) should distinguish $|0_L\rangle$ from $|1_L\rangle$ by sign — a logical $\bar{Z}$.

In [7]:
X_bar = pauli_string_matrix("ZIIZIIZII")  # Z0 Z3 Z6
Z_bar = pauli_string_matrix("XXXIIIIII")  # X0 X1 X2

print("X_bar |0_L> == |1_L> (up to sign):", np.allclose(X_bar @ state_0L, state_1L) or np.allclose(X_bar @ state_0L, -state_1L))
print("X_bar |1_L> == |0_L> (up to sign):", np.allclose(X_bar @ state_1L, state_0L) or np.allclose(X_bar @ state_1L, -state_0L))
print("Z_bar |0_L> eigenvalue:", np.vdot(state_0L, Z_bar @ state_0L).real)
print("Z_bar |1_L> eigenvalue:", np.vdot(state_1L, Z_bar @ state_1L).real)

X_bar |0_L> == |1_L> (up to sign): True
X_bar |1_L> == |0_L> (up to sign): True
Z_bar |0_L> eigenvalue: 0.9999999999999999
Z_bar |1_L> eigenvalue: -0.9999999999999999


Both are genuine weight-3 logical operators, so $d \le 3$. To show $d \ge 3$ (no weight-1 or weight-2 operator can act as a nontrivial logical operator), we exhaustively test every weight-1 and weight-2 Pauli string. Two Pauli strings commute exactly when the number of positions where they carry *different, non-identity* letters is even — a plain counting rule, no matrices needed for the filtering step.

In [8]:
from itertools import combinations, product as iproduct

N_QUBITS = 9


def commutes(p, q):
    disagreements = sum(1 for a, b in zip(p, q) if a != "I" and b != "I" and a != b)
    return disagreements % 2 == 0


def all_weight_le(max_weight, n=N_QUBITS):
    strings = []
    for weight in range(1, max_weight + 1):
        for positions in combinations(range(n), weight):
            for letters in iproduct("XYZ", repeat=weight):
                s = ["I"] * n
                for pos, letter in zip(positions, letters):
                    s[pos] = letter
                strings.append("".join(s))
    return strings


def logical_action_is_trivial(pauli_str):
    """True if this operator acts as a scalar (not a genuine logical op) on the code space."""
    P = pauli_string_matrix(pauli_str)
    a00 = np.vdot(state_0L, P @ state_0L)
    a01 = np.vdot(state_0L, P @ state_1L)
    a10 = np.vdot(state_1L, P @ state_0L)
    a11 = np.vdot(state_1L, P @ state_1L)
    off_diagonal_zero = np.isclose(abs(a01), 0) and np.isclose(abs(a10), 0)
    diagonal_equal = np.isclose(a00, a11)
    return off_diagonal_zero and diagonal_equal


candidates = all_weight_le(2)
nontrivial_low_weight_logicals = []
for s in candidates:
    if all(commutes(s, g) for g in shor_generator_strings):
        if not logical_action_is_trivial(s):
            nontrivial_low_weight_logicals.append(s)

print(f"Checked {len(candidates)} weight-1 and weight-2 Pauli strings.")
print("Nontrivial logical operators found at weight <= 2:", nontrivial_low_weight_logicals)
print("Distance is therefore exactly 3:", len(nontrivial_low_weight_logicals) == 0)

Checked 351 weight-1 and weight-2 Pauli strings.
Nontrivial logical operators found at weight <= 2: []
Distance is therefore exactly 3: True


No weight-1 or weight-2 operator escapes detection *and* acts nontrivially on the logical qubit, while we already exhibited weight-3 operators that do. **The Shor code is exactly $[[9,1,3]]$.**

## 6. The CSS construction: quantum codes from classical codes

The **CSS construction** (Calderbank–Shor–Steane) is a general recipe: take a classical linear code's parity-check matrix $H$ (rows = classical parity checks, as in notebook 04) and turn *each row* into two quantum stabilizers — one built entirely from $X$s on that row's support, one entirely from $Z$s.

Row $h_i$ as an $X$-type stabilizer and row $h_j$ as a $Z$-type stabilizer commute exactly when they overlap in an even number of positions, i.e. when $h_i \cdot h_j = 0 \pmod 2$. So the whole construction works — *every* $X$-type generator commutes with *every* $Z$-type generator — precisely when

$$H H^T \equiv 0 \pmod 2.$$

This is a real structural requirement, not automatic: it says the classical code's parity checks must be pairwise orthogonal to each other (including a check being "orthogonal to itself," i.e. having even weight). Shor's code, from Sections 2–5, is a special case of this idea, though built by hand rather than from one classical code.

### Prediction 2

Recall the $[7,4,3]$ Hamming code's parity-check matrix $H$ from notebook 04 has three rows, each of weight 4 (even). **Before running the next cell:** do you expect $HH^T \bmod 2$ to be the zero matrix? (Hint: weight-4, i.e. even-weight, rows are necessary but not sufficient for pairwise orthogonality — the *off-diagonal* dot products between different rows also have to come out even.)

In [9]:
def bits_of(j, width):
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))


# Same construction as notebook 04: H's columns are the binary representations of 1..7.
H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))

H_np = np.array(H)
HHt_mod2 = (H_np @ H_np.T) % 2
print("H =")
for row in H:
    print(row)
print("H H^T mod 2 =")
print(HHt_mod2)
print("All zero (checks are pairwise orthogonal):", np.all(HHt_mod2 == 0))

H =
(0, 0, 0, 1, 1, 1, 1)
(0, 1, 1, 0, 0, 1, 1)
(1, 0, 1, 0, 1, 0, 1)
H H^T mod 2 =
[[0 0 0]
 [0 0 0]
 [0 0 0]]
All zero (checks are pairwise orthogonal): True


The Hamming code's parity-check matrix satisfies $HH^T \equiv 0 \pmod 2$ — it is, in the standard terminology, **dual-containing** (its dual code sits inside itself). That single fact is the entire reason the *same* $H$ can generate both the $X$-type and $Z$-type stabilizers of a quantum code — no second classical code is needed.

## 7. The Steane code: CSS applied to the Hamming code

Turn each of $H$'s 3 rows into one $X$-type and one $Z$-type stabilizer on 7 qubits: 6 generators total, on $n=7$ qubits, giving $k = n - 6 = 1$ logical qubit — the Steane $[[7,1,3]]$ code, "the quantum version of the Hamming code."

In [10]:
def row_to_pauli_string(row, letter, n=7):
    return "".join(letter if bit else "I" for bit in row)


steane_x_strings = [row_to_pauli_string(row, "X") for row in H]
steane_z_strings = [row_to_pauli_string(row, "Z") for row in H]
steane_generator_strings = steane_x_strings + steane_z_strings

print("X-type generators:", steane_x_strings)
print("Z-type generators:", steane_z_strings)

X-type generators: ['IIIXXXX', 'IXXIIXX', 'XIXIXIX']
Z-type generators: ['IIIZZZZ', 'IZZIIZZ', 'ZIZIZIZ']


### Verification: direct commutation check

Confirm every $X$-type generator commutes with every $Z$-type generator, using actual $128 \times 128$ matrices this time (not just the combinatorial shortcut) — a direct, literal check of Section 1's stabilizer definition.

In [11]:
steane_x_matrices = [pauli_string_matrix(s) for s in steane_x_strings]
steane_z_matrices = [pauli_string_matrix(s) for s in steane_z_strings]

all_commute = True
for sx, Sx in zip(steane_x_strings, steane_x_matrices):
    for sz, Sz in zip(steane_z_strings, steane_z_matrices):
        commute = np.allclose(Sx @ Sz, Sz @ Sx)
        all_commute &= commute
        print(f"[{sx}, {sz}] commute: {commute}")

print("\nEvery X-type generator commutes with every Z-type generator:", all_commute)

[IIIXXXX, IIIZZZZ] commute: True
[IIIXXXX, IZZIIZZ] commute: True
[IIIXXXX, ZIZIZIZ] commute: True
[IXXIIXX, IIIZZZZ] commute: True
[IXXIIXX, IZZIIZZ] commute: True
[IXXIIXX, ZIZIZIZ] commute: True
[XIXIXIX, IIIZZZZ] commute: True
[XIXIXIX, IZZIIZZ] commute: True
[XIXIXIX, ZIZIZIZ] commute: True

Every X-type generator commutes with every Z-type generator: True


In [12]:
steane_generators = steane_x_matrices + steane_z_matrices
dim7 = 128
identity7 = np.eye(dim7, dtype=complex)
projector7 = identity7.copy()
for S in steane_generators:
    projector7 = projector7 @ ((identity7 + S) / 2)

rank7 = np.linalg.matrix_rank(projector7, tol=1e-8)
print("Rank of the joint +1-eigenspace projector:", rank7, " (expected 2^k = 2, so k = 1: the Steane code encodes exactly 1 logical qubit)")

Rank of the joint +1-eigenspace projector: 2  (expected 2^k = 2, so k = 1: the Steane code encodes exactly 1 logical qubit)


### Prediction 3

Notebook 04 showed that a classical single-bit error's syndrome, under this exact $H$, directly names the error position as a binary number. **Before running the next cell:** for a single $X$ error on qubit 3 (0-indexed) of the Steane code, which stabilizers should fire — the $X$-type or the $Z$-type ones? What 3-bit syndrome (from those checks) do you expect, and how does it relate to notebook 04's syndrome table?

In [13]:
def steane_syndrome(error_string):
    E = pauli_string_matrix(error_string)
    # Any nonzero codeword works as a reference codespace state; use the all-zero-message Steane |0_L>-adjacent
    # basis vector |0000000> projected onto the code space via the projector built above.
    reference = projector7 @ basis_state((0, 0, 0, 0, 0, 0, 0))
    reference = reference / np.linalg.norm(reference)
    corrupted = E @ reference
    z_syndrome = tuple(0 if np.isclose(np.vdot(corrupted, Sz @ corrupted).real, 1) else 1 for Sz in steane_z_matrices)
    x_syndrome = tuple(0 if np.isclose(np.vdot(corrupted, Sx @ corrupted).real, 1) else 1 for Sx in steane_x_matrices)
    return x_syndrome, z_syndrome


for qubit in range(7):
    error_string = "".join("X" if i == qubit else "I" for i in range(7))
    x_syn, z_syn = steane_syndrome(error_string)
    print(f"X error on qubit {qubit}: X-type syndrome (should be all 0) = {x_syn}, Z-type syndrome (locates it) = {z_syn}")

X error on qubit 0: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 0, 1)
X error on qubit 1: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 1, 0)
X error on qubit 2: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 1, 1)
X error on qubit 3: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 0, 0)
X error on qubit 4: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 0, 1)
X error on qubit 5: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 1, 0)
X error on qubit 6: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 1, 1)


An $X$ error is only visible to the $Z$-type checks (same-letter Paulis commute), and — because the $Z$-type generators are built from the very same $H$ rows as notebook 04 — the resulting 3-bit syndrome is exactly notebook 04's classical syndrome for an error at that position. A $Z$ error would instead show up only in the $X$-type syndrome, by the mirror argument. The Steane code is, quite literally, two independent copies of the classical Hamming decoder — one watching for $X$ errors, one for $Z$ errors — bolted together by the CSS construction.

## Exercises

**1.** Write `demangle_degenerate_z_error(state)` that takes `state_0L` corrupted by a $Z$ error on any one of block C's three qubits (positions 6, 7, or 8) and applies a *fixed* correction (always `Z6`) regardless of which of the three actually happened. Confirm it restores `state_0L` exactly (up to global phase) in all three cases — this is Section 4's degeneracy argument, made into a working corrector.

**2.** Write `verify_css_condition(H)` that takes any binary parity-check matrix (as a tuple of tuples, like `H` in Section 6) and returns whether $HH^T \equiv 0 \pmod 2$. Try it on `H` from this notebook (should be `True`), and then on a parity-check matrix you construct for the $[3,1,3]$ repetition code from notebook 02 (a single row, `(1,1,0)` say) — does the repetition code's check matrix satisfy the CSS condition? What would that imply about building a "repetition-code CSS quantum code" the same way this notebook built the Steane code?

In [14]:
def demangle_degenerate_z_error(state):
    pass  # TODO: implement, then remove this stub body


def verify_css_condition(H):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# for qubit in (6, 7, 8):
#     error_string = "".join("Z" if i == qubit else "I" for i in range(9))
#     corrupted = pauli_string_matrix(error_string) @ state_0L
#     print(qubit, np.allclose(demangle_degenerate_z_error(corrupted), state_0L))
# print(verify_css_condition(H))